# 2026年11月5日当前注释版本

本文件保留旧冻结流程。当前入口05_annotation_review_20261105.ipynb及版本化修订H5AD。原05簇明确保存为annotation_cluster，不能混用原06的leiden。运行旧历史代码仍会重建旧口径。


## 1. 参数与标记基因


In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sc.settings.verbosity = 0
sc.settings.figdir = "../figures/annotation"

In [2]:
PANELS = {
    "t_cell": ["Cd3d", "Cd3e", "Cd3g", "Lck", "Lat", "Cd247", "Themis"],
    "b_cell": ["Cd79a", "Cd79b", "Cd19", "Ms4a1", "Pax5", "Cd22", "Bank1", "Cd37", "Cd40", "Spib"],
    "nk_cell": ["Ncr1", "Klrd1", "Klrb1c", "Klrk1", "Eomes", "Il2rb", "Nkg7", "Prf1", "Gzmb", "Ccl5"],
    "monocyte": ["Ly6c2", "Ccr2", "Vcan", "Plac8", "Lyz2", "Itgam", "Csf1r", "Sell", "S100a8", "S100a9"],
    "dendritic": ["Flt3", "Zbtb46", "Itgax", "H2-Ab1", "H2-Aa", "Cd74", "Clec10a", "Xcr1", "Clec9a", "Sirpa"],
    "endothelial": ["Pecam1", "Cdh5", "Kdr", "Flt1", "Vwf", "Emcn", "Cldn5", "Esam", "Egfl7", "Tek", "Robo4", "Klf2"],
    "macrophage": ["C1qa", "C1qb", "C1qc", "Mertk", "Fcgr1", "Cd68", "Aif1", "Ctss", "Tyrobp",
                   "Apoe", "Lgals3", "Marco", "Pparg"],
    "fibroblast": ["Pdgfra", "Col1a1", "Col1a2", "Col3a1", "Dcn", "Lum",
                   "Col5a1", "Col6a1", "Col6a2", "Col14a1", "Dpt", "Pi16", "C7"],
    "neutrophil": ["Ly6g", "Csf3r", "Retnlg", "Ngp", "Camp", "Ltf", "Mmp8", "Mmp9",
                   "S100a8", "S100a9", "Mpo", "Elane", "Cebpe", "Ms4a3"],
    "eosinophil": ["Epx", "Prg2", "Prg3", "Siglecf", "Ccr3", "Alox15",
                   "Ear1", "Ear2", "Ear6", "Il5ra", "Gata1",
                   "Itgam", "Ptprc", "Itga4"],
    "basophil": ["Mcpt8", "Cd200r3", "Clec12a", "Fcer1a", "Ms4a2", "Hdc",
                 "Prss34", "Alox5ap", "Il4", "Ccl3", "Ccl4", "Gata2"],
}
SUBTYPE_MARKERS = {
    "classical_monocyte": ["Ly6c2", "Ccr2", "Sell", "Vcan"],
    "nonclassical_monocyte": ["Fcgr3", "Cx3cr1", "Nr4a1", "Treml4"],
    "cDC1": ["Xcr1", "Clec9a", "Batf3", "Irf8"],
    "cDC2": ["Clec10a", "Sirpa", "Irf4"],
    "cytotoxic_NK": ["Ncr1", "Klrd1", "Nkg7", "Prf1", "Gzmb"],
    "plasma_cell": ["Jchain", "Mzb1", "Xbp1", "Sdc1"],
}
PLASMA_NEW = ["Prdm1", "Irf4", "Derl3", "Tnfrsf17", "Slamf7"]
PLASMA_SPECIFIC = ["Derl3", "Tnfrsf17", "Slamf7", "Sdc1"]
PLASMA_EXTRA = ["Prdm1", "Irf4", "Xbp1", "Cd79a"]

## 2. 表达证据


In [3]:
adata = sc.read_h5ad("../data_interim/dimensionality_reduced.h5ad", backed="r")

allgenes = [g for v in list(PANELS.values()) + list(SUBTYPE_MARKERS.values()) for g in v]
allgenes += PLASMA_NEW + ["Sdc1", "Xbp1", "Cd79a", "Cd79b", "Ms4a1"]
allgenes = list(dict.fromkeys(allgenes))
present = [g for g in allgenes if g in adata.var_names]

slim = adata[:, present].to_memory()
X = slim.X
X = np.asarray(X.toarray() if hasattr(X, "toarray") else X, dtype=np.float32)

raw_cl = adata.obs["leiden"].astype(str).values
order = sorted(pd.unique(raw_cl), key=int)
slim.obs["leiden"] = pd.Categorical(raw_cl, categories=order, ordered=True)
slim.obs["tissue"] = adata.obs["tissue"].astype(str).values
slim.obs["mouse.id"] = adata.obs["mouse.id"].astype(str).values
slim.obs["n_genes"] = adata.obs["n_genes"].values
slim.obs["pct_counts_mt"] = adata.obs["pct_counts_mt"].values
slim.obsm["X_umap"] = np.asarray(adata.obsm["X_umap"])

In [4]:
df = pd.DataFrame(X > 0, columns=present, dtype=bool)
df["cl"] = slim.obs["leiden"].astype(str).values
pct = df.groupby("cl", observed=True)[present].mean()
del df

In [5]:
sup = pd.DataFrame({k: (pct[[g for g in v if g in present]] > 0.30).sum(axis=1)
                    for k, v in PANELS.items()})
med = pd.DataFrame({k: pct[[g for g in v if g in present]].median(axis=1)
                    for k, v in PANELS.items()})

V = sup.values
o = np.argsort(-V, axis=1)
t1 = sup.columns[o[:, 0]]
t2 = sup.columns[o[:, 1]]
s1 = V[np.arange(len(V)), o[:, 0]]
s2 = V[np.arange(len(V)), o[:, 1]]

ev = pd.DataFrame({
    "n_cells": slim.obs["leiden"].value_counts().reindex(sup.index).astype(int),
    "top1": t1, "s1": s1, "top2": t2, "s2": s2, "margin": s1 - s2,
    "med1": [round(float(med.loc[c, t]), 2) for c, t in zip(sup.index, t1)],
}, index=sup.index)
ev["ng_median"] = slim.obs.groupby("leiden", observed=True)["n_genes"].median().astype(int)
ev["auto"] = np.where((ev.s1 >= 5) & (ev.margin >= 2), ev.top1, "REVIEW")

## 3. 注释与复核


In [6]:
MANUAL = {
    "0": ("endothelial", "Pecam1/Cdh5/Kdr/Flt1/Emcn/Cldn5/Esam/Egfl7 等 11 个阳性"),
    "1": ("endothelial", "同上 11 个 + Vwf"),
    "19": ("endothelial", "10 个阳性，Kidney"),
    "39": ("endothelial", "11 个阳性，Liver"),
    "41": ("endothelial", "12/12 全阳性，Lung"),
    "4": ("fibroblast", "fibroblast panel 12/13（Pdgfra/Col1a1/Col1a2/Col3a1/Dcn/Lum/Dpt/Pi16）；脂肪组织"),
    "6": ("fibroblast", "Col1a2/Col3a1/Dcn/Col6a1 4 个（跨组织间质）；support<5 故人工确认"),
    "17": ("fibroblast", "9 个阳性，Lung"),
    "20": ("fibroblast", "12/13 阳性，Heart 成纤维细胞"),
    "2": ("b_cell", "Cd79a/Cd79b/Cd19/Ms4a1/Cd22/Bank1/Cd37/Spib 8 个"),
    "3": ("b_cell", "同上 8 个"),
    "18": ("b_cell", "同上 8 个"),
    "5": ("t_cell", "Cd3d/e/g+Lck/Lat/Cd247 6 个"),
    "11": ("t_cell", "6 个 CD3 复合体基因 + 细胞毒程序(Klrd1/Nkg7/Prf1)，判 T 不判 NK"),
    "35": ("t_cell", "同 c11"),
    "12": ("nk_cell", "Ncr1/Klrd1/Klrb1c/Klrk1/Eomes/Il2rb/Nkg7/Prf1 等 10/10"),
    "13": ("dendritic", "Flt3/Itgax/Clec9a/Sirpa+MHC-II 7 个；DC/巨噬细胞交界，取 DC"),
    "14": ("macrophage", "Fcgr1/C1qa/b/c/Cd68/Aif1/Ctss/Tyrobp 等 10 个，Kidney"),
    "30": ("macrophage", "10 个，med1=1.00，Liver(Kupffer 候选)"),
    "31": ("macrophage", "9 个，Lung"),
    "38": ("macrophage", "10 个，Liver"),
    "40": ("macrophage", "Marco 0.44/Pparg 0.37/Mrc1 0.96/Itgax 0.72，C1q 阴性 → 肺泡巨噬；补基因后 macrophage 7 : monocyte 5"),
    "9": ("monocyte", "Ly6c2 0.69/Ccr2 0.69/Csf1r 0.91/Lyz2 1.00，无 Ngp/Camp，Lung"),
    "16": ("monocyte", "Ly6c2 0.85/Ccr2 0.73/Csf1r 0.57/Plac8 0.99，无 Ngp/Camp，Lung"),
    "32": ("neutrophil", "Ngp/Camp 1.00、Retnlg 0.97、Ltf 0.94、Mmp8 0.77、Mmp9 0.90（成熟中性粒）"),
    "33": ("neutrophil", "Ngp/Camp 1.00 + Mpo 0.62/Elane 0.78/Cebpe 0.99/Ms4a3 0.84（未成熟/粒系祖）"),
    "10": ("neutrophil", "Ngp 0.94/Camp 0.95/Mpo 0.51/Elane 0.68（未成熟中性粒）"),
    "21": ("neutrophil", "Ngp 0.83/Camp 0.85/Mpo 0.45"),
    "24": ("neutrophil", "Ngp 0.94/Camp 0.95；n_genes 中位数 619，低复杂度，置信 low"),
    "25": ("neutrophil", "Ngp 0.91/Camp 0.91"),
    "26": ("neutrophil", "Ngp/Camp 0.99、Mmp8 0.89、Mmp9 0.93"),
    "27": ("neutrophil", "Ngp 0.95/Camp 0.96"),
    "22": ("basophil", "Mcpt8 0.79/Cd200r3 0.92/Fcer1a 0.93/Hdc 1.00/Alox5ap 0.99/Gata2 0.97，Lung n=120"),
    "8": ("plasma_cell", "Derl3 0.96/Sdc1 0.94/Slamf7 0.83/Tnfrsf17 0.65，组合证据 97.9%"),
    "23": ("ambiguous", "monocyte 7 : neutrophil 7；Ngp 0.33/Camp<0.2 但 Csf3r 0.98/Mmp8 0.84，未成熟粒/单核交界"),
    "7": ("unassigned", "macrophage 2 / endothelial 2，med 0.04（Kidney 上皮）"),
    "15": ("unassigned", "endothelial 1 / macrophage 1；n_genes 434 低质量"),
    "28": ("unassigned", "dendritic 2（仅 MHC-II）/ macrophage 1，med 0.00（Kidney 上皮）"),
    "29": ("unassigned", "macrophage 2（Apoe/Lgals3）；Tnnt2 1.00 为心肌细胞"),
    "34": ("unassigned", "macrophage 2 / dendritic 2，med 0.07"),
    "36": ("unassigned", "macrophage 1（仅 Apoe），Liver（肝细胞）"),
    "37": ("unassigned", "macrophage 1（仅 Apoe），Liver（肝细胞）"),
}

missing_cl = [c for c in ev.index if c not in MANUAL]
extra_cl = [c for c in MANUAL if c not in ev.index]
assert not missing_cl
assert not extra_cl

slim.obs["cell_type"] = slim.obs["leiden"].map({c: v[0] for c, v in MANUAL.items()}).astype(str)
slim.obs["auto_label"] = slim.obs["leiden"].map(ev["auto"].to_dict()).astype(str)

In [7]:
ev["manual"] = [MANUAL[c][0] for c in ev.index]
ev["rationale"] = [MANUAL[c][1] for c in ev.index]
ev["tissue_top"] = [
    "/".join(f"{k} {v:.0%}" for k, v in
             slim.obs.loc[slim.obs.leiden == c, "tissue"].value_counts(normalize=True).head(2).items())
    for c in ev.index]
ev["n_mice"] = [slim.obs.loc[slim.obs.leiden == c, "mouse.id"].nunique() for c in ev.index]
ev["n_cells"] = [int((slim.obs.leiden == c).sum()) for c in ev.index]

def conf(c):
    lab = MANUAL[c][0]
    if lab in ("ambiguous", "unassigned"):
        return "na"
    if lab == "plasma_cell":
        return "high"
    if c in ("24", "15"):
        return "low"
    if ev.loc[c, "s1"] >= 8 and ev.loc[c, "margin"] >= 3:
        return "high"
    return "medium"

ev["confidence"] = [conf(c) for c in ev.index]
slim.obs["annotation_confidence"] = slim.obs["leiden"].map(ev["confidence"].to_dict()).astype(str)

audit = ev[["n_cells", "n_mice", "tissue_top", "ng_median", "top1", "s1", "top2", "s2",
            "margin", "med1", "auto", "manual", "confidence", "rationale"]]
audit.index.name = "cluster"
audit.to_csv("../results/cell_annotation/v2/cluster_annotation.tsv", sep="\t")

## 4. 注释展示


In [8]:
GROUPS = {
    "lymphoid": {"t_cell": PANELS["t_cell"], "b_cell": PANELS["b_cell"],
                 "nk_cell": PANELS["nk_cell"], "plasma": PLASMA_SPECIFIC + ["Prdm1", "Irf4", "Xbp1"]},
    "myeloid": {"monocyte": PANELS["monocyte"], "dendritic": PANELS["dendritic"],
                "macrophage": [g for g in PANELS["macrophage"] if g in present]},
    "structural": {"endothelial": PANELS["endothelial"],
                   "fibroblast": PANELS["fibroblast"]},
    "granulocyte": {"neutrophil": PANELS["neutrophil"],
                    "eosinophil": PANELS["eosinophil"],
                    "basophil": PANELS["basophil"]},
}

for name, vn in GROUPS.items():
    vn = {k: [g for g in v if g in present] for k, v in vn.items()}
    sc.pl.dotplot(slim, vn, groupby="leiden", standard_scale="var", cmap="Reds",figsize=(13, 11))
    plt.savefig(f"../figures/annotation/05_dotplot_{name}.png", dpi=160, bbox_inches="tight")
    plt.close()

c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
slim.obs["cell_type"] = pd.Categorical(
    slim.obs["cell_type"],
    categories=["endothelial", "fibroblast", "macrophage", "monocyte", "neutrophil", "basophil", "dendritic", "t_cell",
                "nk_cell", "b_cell", "plasma_cell", "ambiguous", "unassigned"])

sc.pl.umap(slim, color="cell_type", frameon=False, palette="tab20",size=2, alpha=0.8, title="cell_type")
plt.savefig("../figures/annotation/05_umap_cell_type.png", dpi=200, bbox_inches="tight")
plt.close()

sc.pl.umap(slim, color="annotation_confidence", frameon=False, size=2, alpha=0.8, title="annotation confidence")
plt.savefig("../figures/annotation/05_umap_confidence.png", dpi=200, bbox_inches="tight")
plt.close()

sc.pl.umap(slim, color="tissue", frameon=False, size=2, alpha=0.8,title="tissue")
plt.savefig("../figures/annotation/05_umap_tissue.png", dpi=200, bbox_inches="tight")
plt.close()

c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
PLOT = {
    "endothelial": PANELS["endothelial"],
    "fibroblast": PANELS["fibroblast"],
    "macrophage": [g for g in PANELS["macrophage"] if g in present],
    "monocyte": PANELS["monocyte"],
    "dendritic": PANELS["dendritic"],
    "t_cell": PANELS["t_cell"],
    "nk_cell": PANELS["nk_cell"],
    "b_cell": PANELS["b_cell"],
    "plasma_cell": PLASMA_SPECIFIC + ["Prdm1", "Irf4", "Xbp1"],
    "neutrophil": PANELS["neutrophil"],
    "eosinophil": PANELS["eosinophil"],
    "basophil": PANELS["basophil"],
}

for ct, genes in PLOT.items():
    genes = [g for g in genes if g in present]
    sc.pl.umap(slim, color=genes, ncols=4, vmax="p99", cmap="Reds",frameon=False, size=3)
    plt.suptitle(f"{ct} markers", y=1.01, fontsize=13)
    plt.savefig(f"../figures/annotation/05_umap_markers_{ct}.png", dpi=150, bbox_inches="tight")
    plt.close()

c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\scanpy\plotting\_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\a

## 5. 细胞筛选与导出


In [11]:
slim = slim[~slim.obs["cell_type"].astype(str).isin(["unassigned", "ambiguous"])].copy()

mct= {
    "endothelial": "endothelial",
    "fibroblast": "fibroblast",
    "monocyte": "myeloid", "macrophage": "myeloid", "dendritic": "myeloid",
    "neutrophil": "myeloid", "basophil": "myeloid", "eosinophil": "myeloid",
    "t_cell": "lymphoid", "b_cell": "lymphoid", "nk_cell": "lymphoid",
    "plasma_cell": "lymphoid",
}
slim.obs["mct"] = slim.obs["cell_type"].astype(str).map(mct)
assert slim.obs["mct"].notna().all()
slim.obs["mct"] = pd.Categorical(slim.obs["mct"],
                                categories=["endothelial", "fibroblast", "myeloid", "lymphoid"],
                                ordered=True)

In [12]:
lbl = slim.obs[["leiden", "tissue", "mouse.id", "cell_type", "mct","annotation_confidence", "auto_label"]].copy()
lbl.index.name = "cell"
lbl.to_csv("../results/cell_annotation/cell_labels.tsv.gz", sep="\t", compression="gzip")

slim.write_h5ad("../data_interim/cell_type_annotated.h5ad")

summary = (lbl.groupby(["mct", "cell_type"], observed=True)
           .agg(n_cells=("leiden", "size"), n_mice=("mouse.id", "nunique"))
           .reset_index().sort_values("n_cells", ascending=False))
summary.to_csv("../results/cell_annotation/cell_type_summary.tsv", sep="\t", index=False)

# 读取全基因矩阵。
adata_full = sc.read_h5ad("../data_interim/dimensionality_reduced.h5ad")

# 保留已筛选细胞。
adata_full = adata_full[slim.obs_names].copy()

# 同步注释列。
adata_full.obs['cell_type'] = slim.obs['cell_type']
adata_full.obs['mct'] = slim.obs['mct']
adata_full.obs['annotation_confidence'] = slim.obs['annotation_confidence']
adata_full.obs['auto_label'] = slim.obs['auto_label']

# 保存全基因注释对象。
adata_full.write_h5ad("../data_interim/cell_type_annotated.h5ad")

## 2026-10-30交付补齐 · 当前marker核查、统一标签及120/117队列矩阵
独立读取已有对象导出，不重跑上游拟合；运行目录为notebooks。历史版本保留在整理备份。


In [13]:
import json
import hashlib
import h5py
import numpy as np
import pandas as pd
from anndata.io import read_elem
from scipy import sparse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.rcParams["font.sans-serif"]=["Microsoft YaHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
plt.rcParams["pdf.fonttype"]=42

! powershell.exe -NoProfile -Command "New-Item -ItemType Directory -Force '../results/qc','../figures/qc','../figure_data' | Out-Null"
# Existing 05 marker definitions; labels are read from the frozen object, never reassigned.
marker_panels={"T cell":["Cd3d","Cd3e","Lck","Cd247"],"B cell":["Cd79a","Cd79b","Ms4a1","Pax5"],"NK":["Ncr1","Klrd1","Nkg7","Prf1"],"Plasma":["Derl3","Tnfrsf17","Slamf7","Sdc1"],"Monocyte":["Ly6c2","Ccr2","Lyz2","Csf1r"],"Dendritic":["Flt3","Itgax","H2-Ab1","Cd74"],"Macrophage":["C1qa","C1qb","Mertk","Cd68"],"Neutrophil":["Ly6g","Csf3r","Retnlg","Ngp"],"Eosinophil":["Epx","Prg2","Siglecf","Ccr3"],"Basophil":["Mcpt8","Fcer1a","Ms4a2","Hdc"]}
with h5py.File("../data_processed/final_data.h5ad","r") as handle:
    marker_obs=read_elem(handle["obs"]).copy();marker_var=read_elem(handle["var"])
    marker_genes=list(dict.fromkeys(g for panel in marker_panels.values() for g in panel))
    marker_present=[g for g in marker_genes if g in marker_var.index];selected_cols=marker_var.index.get_indexer(marker_present)
    marker_values=np.empty((len(marker_obs),len(marker_present)),dtype=np.float32)
    for start in range(0,len(marker_obs),2048):
        stop=min(start+2048,len(marker_obs));ptr=np.asarray(handle["X/indptr"][start:stop+1]);left,right=int(ptr[0]),int(ptr[-1])
        block=sparse.csr_matrix((np.asarray(handle["X/data"][left:right]),np.asarray(handle["X/indices"][left:right]),ptr-left),shape=(stop-start,len(marker_var)))
        marker_values[start:stop]=block[:,selected_cols].toarray()
marker_table=pd.DataFrame(marker_values,columns=marker_present,index=marker_obs.index)
for col in ["mouse.id","tissue","cell_type","mct"]:marker_table[col]=marker_obs[col].astype(str)
sample_means=marker_table.groupby(["mouse.id","tissue","cell_type","mct"],observed=True)[marker_present].mean()
detection_table=pd.DataFrame(marker_values>0,columns=marker_present,index=marker_obs.index)
for col in ["mouse.id","tissue","cell_type","mct"]:detection_table[col]=marker_obs[col].astype(str)
sample_detect=detection_table.groupby(["mouse.id","tissue","cell_type","mct"],observed=True)[marker_present].mean()
expression=sample_means.groupby(["mct","cell_type"],observed=True).mean()
detection=sample_detect.groupby(["mct","cell_type"],observed=True).mean().reindex(expression.index)
expression.to_csv("../figure_data/celltype_marker_mean_log_expression.tsv",sep="\t")
detection.to_csv("../figure_data/celltype_marker_fraction_detected.tsv",sep="\t")
pd.DataFrame([dict(panel=panel,gene=gene,present_in_frozen=gene in marker_present,source="05 existing PANELS / PLASMA_SPECIFIC") for panel,genes in marker_panels.items() for gene in genes]).to_csv("../results/qc/marker_panel_audit.tsv",sep="\t",index=False)
scaled=expression.divide(expression.max(axis=0).replace(0,np.nan),axis=1).fillna(0)
fig,ax=plt.subplots(figsize=(16,7),layout="constrained")
for i in range(len(expression)):
    ax.scatter(np.arange(len(marker_present)),np.repeat(i,len(marker_present)),s=8+170*detection.iloc[i].to_numpy(),c=scaled.iloc[i].to_numpy(),vmin=0,vmax=1,cmap="Reds",edgecolors="#999999",linewidths=.25)
ax.set_xticks(range(len(marker_present)),marker_present,rotation=90,fontsize=8)
ax.set_yticks(range(len(expression)),[a+" | "+b for a,b in expression.index],fontsize=8);ax.invert_yaxis()
ax.set_title("Frozen current annotations | existing marker panels\nColour: mean log expression / gene maximum; size: detected fraction; equal mouse-ID x tissue weight",fontsize=11)
fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0,1),cmap="Reds"),ax=ax,label="Within-gene scaled mean",shrink=.6)
fig.savefig("../figures/qc/celltype_marker_check.pdf");fig.savefig("../figures/qc/celltype_marker_check.png",dpi=140);plt.close(fig)
# The final cell object has 120 observed groups; 117 survive the existing pseudobulk sample rules.
with h5py.File("../data_processed/pseudobulk.h5ad","r") as handle:cohort_pb=read_elem(handle["obs"])
cohort_keys=set(cohort_pb[["mouse.id","tissue","mct"]].astype(str).itertuples(index=False,name=None))
cohort_obs=marker_obs.copy()
for col in ["mouse.id","tissue","mct","age_group","sex"]:cohort_obs[col]=cohort_obs[col].astype(str)
cohort_obs["age_months"]=pd.to_numeric(cohort_obs.age)
cohort_matrix=cohort_obs.groupby(["mouse.id","tissue","mct","age_months","age_group","sex"],observed=True).size().rename("n_cells").reset_index().rename(columns={"mouse.id":"mouse_id"})
cohort_matrix["in_primary_117"]=[(r.mouse_id,r.tissue,r.mct) in cohort_keys for r in cohort_matrix.itertuples()]
cohort_matrix["exploratory_node"]=cohort_matrix.tissue.eq("Liver")&cohort_matrix.mct.eq("lymphoid")
cohort_matrix["sampling_unit"]="mouse.id identifier; composite identifiers retained and audited"
assert len(cohort_matrix)==120 and cohort_matrix.in_primary_117.sum()==117 and cohort_matrix.n_cells.sum()==96569
cohort_matrix.to_csv("../results/qc/final_cohort_matrix.csv",index=False)
cohort_matrix.loc[~cohort_matrix.in_primary_117].to_csv("../results/qc/outside_primary_cohort.csv",index=False)
# Current raw-label to harmonised-label crosswalk; no manual relabeling.
with h5py.File("../data_interim/TMS_checked.h5ad","r") as handle:original_annotation=read_elem(handle["obs"])[["cell_ontology_class"]]
assert cohort_obs.index.isin(original_annotation.index).all()
cohort_obs["cell_ontology_class"]=original_annotation.reindex(cohort_obs.index).cell_ontology_class.astype(str)
crosswalk=cohort_obs.groupby(["cell_ontology_class","cell_type","mct"],observed=True).agg(n_cells=("mouse.id","size"),n_mouse_identifiers=("mouse.id","nunique")).reset_index()
crosswalk.to_csv("../docs/celltype_harmonization.tsv",sep="\t",index=False)
print("Marker audit and current crosswalk complete; final coverage:120 observed groups,117 primary.")


Marker audit and current crosswalk complete; final coverage:120 observed groups,117 primary.
